<a href="https://colab.research.google.com/github/vituhaa/recsys_projects/blob/main/film_recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Обработка данных в Pandas

### Item2item рекомендация

In [5]:
import kagglehub

# Скачиваем датасет
path = kagglehub.dataset_download("trishna8/movielens-100k-dataset")
print("Путь:", path)

Using Colab cache for faster access to the 'movielens-100k-dataset' dataset.
Путь: /kaggle/input/movielens-100k-dataset


In [6]:
import os

for file in os.listdir(path):
    print(f"  {file}")

  ml-100k


In [7]:
print(path)

/kaggle/input/movielens-100k-dataset


In [8]:
from pyspark.sql import SparkSession
import urllib.request

spark = SparkSession.builder.appName("MovieRecommendationSystem").config("spark.driver.memory", "4g").getOrCreate()

ds_path = '/kaggle/input/movielens-100k-dataset/ml-100k'

ratings_file = os.path.join(ds_path, "u.data")
ratings = spark.read.csv(ratings_file, sep="\t",
                         schema="userId INT, movieId INT, rating FLOAT, timestamp LONG")

movies_file = os.path.join(ds_path, "u.item")
movies = spark.read.csv(movies_file, sep="|",
                         schema="movieId INT, title STRING")

In [9]:
ratings.show(5)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|   196|    242|   3.0|881250949|
|   186|    302|   3.0|891717742|
|    22|    377|   1.0|878887116|
|   244|     51|   2.0|880606923|
|   166|    346|   1.0|886397596|
+------+-------+------+---------+
only showing top 5 rows


In [ ]:
movies.show(5)

In [10]:
ratings = ratings.dropna().dropDuplicates()
movies = movies.dropna().dropDuplicates()

Разбиение данных на выборки

In [11]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window = (
    Window
    .partitionBy("userId")
    .orderBy("timestamp")
)

ratings_numbered = (
    ratings
    .withColumn(
        "row_num",
        F.row_number().over(window)
    )
    .withColumn(
        "user_count",
        F.count("*").over(
            Window.partitionBy("userId")
        )
    )
)

ratings_numbered = ratings_numbered.withColumn(
    "split_point",
    F.floor(F.col("user_count") * 0.8)
)

train = (
    ratings_numbered
    .filter(F.col("row_num") <= F.col("split_point"))
    .drop("row_num", "user_count", "split_point")
)

test = (
    ratings_numbered
    .filter(F.col("row_num") > F.col("split_point"))
    .drop("row_num", "user_count", "split_point")
)

print("Train: ", train.count())
print("Test: ", test.count())

Train:  79619
Test:  20381


In [12]:
train.show(5)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|    168|   5.0|874965478|
|     1|    172|   5.0|874965478|
|     1|    165|   5.0|874965518|
|     1|    156|   4.0|874965556|
|     1|    196|   5.0|874965677|
+------+-------+------+---------+
only showing top 5 rows


In [13]:
item_user_matrix = train.groupBy("movieId").pivot("userId").avg("rating").fillna(0)

In [14]:
from pyspark.sql import functions as F

def cosine_similarity(movie1, movie2, item_user):
  user_cols = [x for x in item_user.columns if x != "movieId"]

  v1 = (
      item_user
      .filter(F.col("movieId") == movie1)
      .select(
          F.array(*[F.col(f"`{c}`") for c in user_cols]).alias("v1")
      )
  )

  v2 = (
      item_user
      .filter(F.col("movieId") == movie2)
      .select(
          F.array(*[F.col(f"`{c}`") for c in user_cols]).alias("v2")
      )
  )

  pair = v1.crossJoin(v2)

  result = pair.select(
      (
          F.aggregate(
              F.zip_with("v1", "v2", lambda x, y: x * y), # zip_with идёт попарно по двум массивам
              F.lit(0.0), # начальное значение суммы
              lambda acc, x: acc + x
          )
          /
          F.sqrt(
              F.aggregate(
                  F.transform("v1", lambda x: x * x), # transform проходит по каждому элементу массива
                  F.lit(0.0),
                  lambda acc, x: acc + x
              )
          )
          /
          F.sqrt(
              F.aggregate(
                  F.transform("v2", lambda x: x * x),
                  F.lit(0.0),
                  lambda acc, x: acc + x
              )
          )
      ).alias("cosine")
  )

  return result

In [15]:
user_cols = [x for x in item_user_matrix.columns if x != "movieId"]
movie_vectors = item_user_matrix.select(
    "movieId",
    F.array(*[F.col(f"`{c}`") for c in user_cols]).alias("vector")
)

In [16]:
movie_vectors.show(5)

+-------+--------------------+
|movieId|              vector|
+-------+--------------------+
|    148|[2.0, 0.0, 0.0, 0...|
|    471|[0.0, 0.0, 0.0, 0...|
|    496|[0.0, 0.0, 0.0, 0...|
|    463|[0.0, 0.0, 0.0, 0...|
|    833|[0.0, 0.0, 0.0, 0...|
+-------+--------------------+
only showing top 5 rows


In [17]:
def find_similar_movies(movie_id, item_user_matrix, top_k=10):
    movie_ids = (
        item_user_matrix
        .filter(F.col("movieId") != movie_id)
        .select("movieId")
        .collect()
    )

    result = None

    for row in movie_ids:
        other_movie = row["movieId"]

        sim = (
            cosine_similarity(movie_id, other_movie, item_user_matrix)
            .withColumn("movieId", F.lit(other_movie))
        )

        if result is None:
            result = sim
        else:
            result = result.union(sim)

    return (
        result
        .orderBy(F.desc("cosine"))
        .limit(top_k)
    )

In [27]:
def recommend_user(user_id, train, item_user, top_k=10):
    # фильмы, которые пользователь уже смотрел
    watched = (
        train
        .filter(F.col("userId") == user_id)
        .select("movieId", "rating")
        .collect()
    )

    print("WATCHED: ", watched)

    if len(watched) == 0:
        return None

    watched_ids = [
        row["movieId"]
        for row in watched
    ]

    recommendations = []
    all_similar = None

    # для каждого просмотренного фильма
    for row in watched:
        print("ROW: ", row)

        movie_id = row["movieId"]
        rating = row["rating"]

        # похожие фильмы
        similar_movies = find_similar_movies(movie_id, item_user_matrix, top_k=20)
        print("SIMILAR: ", similar_movies)

        similar_with_score = similar_movies.withColumn(
            "score",
            F.col("cosine") * rating
        )

        if all_similar is None:
            all_similar = similar_with_score
        else:
            all_similar = all_similar.union(similar_with_score)

    # превращаем в Spark DataFrame
    recs = spark.createDataFrame(
        recommendations,
        ["movieId", "score"]
    )

    # убираем просмотренные фильмы
    recommendations = (
        all_similar
        .filter(~F.col("movieId").isin(watched_ids))  # убираем просмотренные
        .groupBy("movieId")
        .agg(F.sum("score").alias("score"))
        .orderBy(F.desc("score"))
        .limit(top_k)
    )

    return recommendations

In [25]:
test_users = (
    test
    .filter(F.col("rating") >= 4)
    .select("userId")
    .distinct()
    .limit(20)
    .collect()
)

In [ ]:
recommend_user(100, train, item_user_matrix, top_k=10)

In [ ]:
# all_recs = None

# for row in test_users[:10]:
#     user_id = row["userId"]

#     recs = (
#         recommend_user(user_id, train, item_user_matrix, top_k=10)
#         .withColumn("userId", F.lit(user_id))
#     )

#     all_recs = recs if all_recs is None else all_recs.union(recs)

In [ ]:
def calculate_metrics(recommendations, test, k=10):
    relevant = (
        test
        .filter(F.col("rating") >= 4)
        .select("userId", "movieId")
        .distinct()
        .withColumn("relevant", F.lit(1))
    )

    # ранжируем рекомендации внутри каждого пользователя
    w_rank = Window.partitionBy("userId").orderBy(F.desc("score"))

    ranked = (
        recommendations
        .withColumn("rank", F.row_number().over(w_rank))
        .filter(F.col("rank") <= k)
        .join(relevant, ["userId", "movieId"], "left")
        .fillna(0, subset=["relevant"])
    )

    # накопленное количество попаданий
    w = (
        Window
        .partitionBy("userId")
        .orderBy("rank")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )

    ranked = (
        ranked
        .withColumn(
            "cum_hits",
            F.sum("relevant").over(w)
        )
        .withColumn(
            "precision_at_rank",
            F.col("cum_hits") / F.col("rank")
        )
    )

    # сколько релевантных фильмов у каждого пользователя в test
    true_count = (
        relevant
        .groupBy("userId")
        .agg(F.count("*").alias("n_relevant"))
    )

    # считаем метрики для каждого пользователя
    user_metrics = (
        ranked
        .groupBy("userId")
        .agg(
            F.sum("relevant").alias("hits"),

            F.sum(
                F.when(
                    F.col("relevant") == 1,
                    F.col("precision_at_rank")
                ).otherwise(0)
            ).alias("ap_sum")
        )
        .join(true_count, "userId")
        .withColumn(
            "precision",
            F.col("hits") / F.lit(k)
        )
        .withColumn(
            "recall",
            F.col("hits") / F.col("n_relevant")
        )
        .withColumn(
            "ap",
            F.col("ap_sum") /
            F.least(F.col("n_relevant"), F.lit(k))
        )
    )

    # усредняем по пользователям
    metrics = user_metrics.agg(
        F.avg("precision").alias(f"Precision@{k}"),
        F.avg("recall").alias(f"Recall@{k}"),
        F.avg("ap").alias(f"MAP@{k}")
    )

    return metrics

### User2user рекомендация на Pandas

# ALS

In [30]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    nonnegative=True,
    coldStartStrategy="drop",
    rank=10,
    maxIter=10,
    regParam=0.1
)

model = als.fit(train)

In [32]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)
print(f"RMSE on test data = {rmse:.3f}")

RMSE on test data = 0.969


In [33]:
# Top 5 recommendations for each user
user_recs = model.recommendForAllUsers(5)
user_recs.show(5, truncate=False)

+------+---------------------------------------------------------------------------------------------+
|userId|recommendations                                                                              |
+------+---------------------------------------------------------------------------------------------+
|1     |[{1463, 5.260474}, {1449, 4.9416327}, {114, 4.8674755}, {119, 4.8645644}, {1367, 4.785984}]  |
|2     |[{1104, 4.925233}, {1449, 4.7305183}, {1463, 4.7209973}, {868, 4.7209153}, {1642, 4.7004666}]|
|3     |[{1268, 4.097015}, {253, 4.0625114}, {793, 4.0039835}, {1159, 3.6510072}, {1344, 3.6398864}] |
|4     |[{1639, 6.4465766}, {1642, 6.2219324}, {1275, 6.1804905}, {904, 6.118702}, {1466, 6.110317}] |
|5     |[{1463, 4.7830462}, {1500, 4.573825}, {1589, 4.4506016}, {50, 4.352107}, {168, 4.2781806}]   |
+------+---------------------------------------------------------------------------------------------+
only showing top 5 rows


In [34]:
user_subset = ratings.select("userId").distinct().limit(3)
model.recommendForUserSubset(user_subset, 5).show(truncate=False)

+------+-------------------------------------------------------------------------------------------+
|userId|recommendations                                                                            |
+------+-------------------------------------------------------------------------------------------+
|471   |[{1344, 5.627137}, {1022, 5.1310444}, {718, 4.8756046}, {668, 4.847476}, {1260, 4.804145}] |
|463   |[{1463, 4.6098065}, {1639, 4.481413}, {1642, 4.3756757}, {113, 4.350707}, {1159, 4.288106}]|
|148   |[{1463, 5.4304404}, {1589, 5.0934944}, {1169, 4.946422}, {114, 4.909092}, {1500, 4.876085}]|
+------+-------------------------------------------------------------------------------------------+



# Content-Based

# Обработка данных в PySpark + Baseline (неперсонализированная рекомендация)

In [1]:
movies.show() # первые 20 строк

NameError: name 'movies' is not defined

In [ ]:
movies.printSchema() # схема данных

In [ ]:
movies_pd = movies.limit(10).toPandas() # перевод в pandas dataframe
movies_pd

In [ ]:
movies.select("genres","title").show(10) # просмотр нужных столбцов

In [ ]:
movies.select("genres").distinct().show() # просмотр уникальных

In [ ]:
ratings.describe("rating").show() # Статистика по числовым колонкам

In [ ]:
from pyspark.sql.functions import to_date, year, col

# timestamp в формате 'YYYY-MM-DD HH:MM:SS'
ratings.select(year(to_date(col("timestamp"))).alias("year")) \
       .distinct() \
       .orderBy("year") \
       .show(50)

In [ ]:
ratings.sample(fraction=0.1, seed=42).show(10) # выборка

In [ ]:
ratings.filter(ratings.rating > 4).explain()

In [ ]:
print("Количество пользователей:")
ratings.select("userId").distinct().count()

In [ ]:
print("Количество фильмов:")
ratings.select("movieId").distinct().count()

In [ ]:
from pyspark.sql.functions import col, avg, count, desc

popular_list = (
    train
    .groupBy("movieId")
    .agg(
        avg("rating").alias("avg_rating"),
        count("userId").alias("rating_count")
    )
)

In [ ]:
# добавляем названия фильмов
popular_with_titles = (
    popular_list
    .join(movies.select("movieId", "title"), on="movieId", how="inner")
    .orderBy(desc("rating_count"))
)

print("Самые популярные фильмы с названиями:")
popular_with_titles.select("title", "rating_count", "avg_rating").show(10, truncate=False)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = (
    test
    .join(popular_list.select("movieId", "avg_rating"), on="movieId", how="left")
    .fillna({"avg_rating": 3.0})
)

# оценка модели
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="avg_rating"
)

rmse = evaluator.evaluate(predictions)
print(f"RMSE бейзлайн модели: {rmse:.4f}")

# функция для рекомендаций
def recommend_popular(user_id, n=10):
    user_movies = ratings.filter(col("userId") == user_id).select("movieId")
    recommendations = (
        popular_list
        .join(movies.select("movieId", "title"), on="movieId")
        .join(user_movies, on="movieId", how="left_anti")
        .orderBy(desc("rating_count"))
        .select("title", "avg_rating", "rating_count")
        .limit(n)
    )
    return recommendations

# пример рекомендации
print("Рекомендации для пользователя 1:")
recommend_popular(1, 10).show(truncate=False)